In [19]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Activation
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.regularizers import L1
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import seaborn as sns

ModuleNotFoundError: No module named 'tensorflow.keras'

In [ ]:
# 1. Veriyi yükleme ve ön işleme
(x_train, y_train), (x_test, y_test) = mnist.load_data()
num_classes = 10
indices = [None] * num_classes
for i in range(len(y_train)):
    label = y_train[i]
    if indices[label] is None:
        indices[label] = i
    if all(index is not None for index in indices):
        break

# Görselleri çizdirme
fig, axes = plt.subplots(1, num_classes, figsize=(10, 4))

for i in range(num_classes):
    if indices[i] is not None:
        image = x_train[indices[i]]
        axes[i].imshow(image, cmap='gray')
        axes[i].set_title(f"Sınıf: {i}")
        axes[i].axis('off')  # Eksenleri kapatma

plt.tight_layout()
plt.show()

x_train = x_train.reshape(60000, 784).astype('float32') / 255
x_test = x_test.reshape(10000, 784).astype('float32') / 255
y_train_categorical = to_categorical(y_train, num_classes)
y_test_categorical = to_categorical(y_test, num_classes)

In [ ]:
model = Sequential([
    Dense(128, input_shape=(784,)),
    BatchNormalization(),
    Activation('relu'),
    Dropout(0.3),

    Dense(64),
    BatchNormalization(),
    Activation('relu'),
    Dropout(0.3),

    Dense(32,activation='relu', activity_regularizer=L1(0.001)),
    BatchNormalization(),
    Dense(num_classes,activation='softmax')
])

In [ ]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
# 4. Modeli eğitme
batch_size = 128
epochs = 10
history = model.fit(x_train, y_train_categorical, batch_size=batch_size, epochs=epochs, validation_split=0.2,verbose=1)

In [ ]:
#5 Modeli değerlendirme
y_pred_probalities = model.predict(x_test)
y_pred = np.argmax(y_pred_probalities, axis=1)
print(classification_report(y_test, y_pred))

In [ ]:
 # 6. Sonuçları görselleştirme
confusion_mtx = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(confusion_mtx, annot=True, fmt='d', cmap='Blues', xticklabels=range(10), yticklabels=range(10))

In [1]:
#7. Confusion matrixi yazdırma
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

NameError: name 'confusion_matrix' is not defined

In [2]:
# 8. Classfication reportu yazdırma
cr = classification_report(y_test, y_pred)
print("Classification Report:")
print(cr)

NameError: name 'classification_report' is not defined

In [ ]:
# 9. Loss grafiği çizdirme
plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'], label='Eğitim Kaybı')
plt.plot(history.history['val_loss'], label='Doğrulama Kaybı')
plt.title('Eğitim ve Doğrulama Kaybı')
plt.xlabel('Epoch Sayısı')
plt.ylabel('Kayıp')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import numpy as np
from PIL import Image
import tkinter as tk
from tkinter import filedialog

# **ÖNEMLİ:** Model dosyasını yükleyin
model = load_model('el_yazisi_tanima_modeli.h5')

# 1. Kullanıcıdan resim dosyası seçmesini isteyin
root = tk.Tk()
root.withdraw()  # Tkinter penceresini gizle
image_path = filedialog.askopenfilename(title="Bir resim dosyası seçin", filetypes=[("Image files", "*.png;*.jpg;*.jpeg")])

if image_path:  # Eğer bir dosya seçildiyse
    try:
        # 2. Resmi yükleme ve boyutlandırma
        img = image.load_img(image_path, target_size=(28, 28), color_mode='grayscale')

        # 3. Resmi NumPy dizisine dönüştürme
        img_array = image.img_to_array(img)

        # 4. Piksel değerlerini 0-1 arasına normalleştirme
        img_array = img_array / 255.0

        # 5. Resmi modelin beklediği formata getirme
        img_array = img_array.reshape(1, 784)

        # 6. Model ile tahmin yapma
        predictions = model.predict(img_array)

        # 7. Tahmin sonuçlarını yorumlama
        predicted_class_index = np.argmax(predictions)
        confidence = predictions[0][predicted_class_index]

        print(f"Tahmin Edilen Rakam: {predicted_class_index}")
        print(f"Güven Skoru: {confidence:.4f}")

    except Exception as e:
        print(f"Bir hata oluştu: {e}")
else:
    print("Hiçbir dosya seçilmedi.")
